# Tuned GBDT comparison

Confronto multi-seed tra B-score logit, GNN corrente, decoder antisimmetrico e
GBDT tuned. Gli artefatti devono condividere match, target, split e scope.

In [1]:
import os
from pathlib import Path
import sys

START = Path.cwd().resolve()
PROJECT_ROOT = START if (START / "edge_feature_ablation").is_dir() else START.parent
sys.path.insert(0, str(PROJECT_ROOT / "edge_feature_ablation"))
from multi_seed_summary import multi_seed_comparison

In [2]:
TOURNAMENT_SCOPE = os.environ.get(
    "TENNIS_RUN_SCOPE", "slams_masters"
)  # or "slams"
SEEDS = [42, 123, 456, 789, 2026]
RESULTS_ROOT = PROJECT_ROOT / "results" / "frozen_predictions" 

In [3]:
per_seed, aggregate = multi_seed_comparison(
    RESULTS_ROOT,
    scope=TOURNAMENT_SCOPE,
    seeds=SEEDS,
    experiments=[
        "bscore_logit",
        "current_gnn",
        "antisymmetric_decoder",
        "gbdt_tuned",
    ],
)
aggregate

,accuracy_mean,accuracy_std,log_loss_mean,log_loss_std,brier_mean,brier_std,accuracy_wins_vs_current,log_loss_wins_vs_current,brier_wins_vs_current
experiment,,,,,,,,,
antisymmetric_decoder,0.702362,0.002960,0.583427,0.002636,0.198568,0.001009,5,5,5
bscore_logit,0.707312,0.002305,0.577825,0.000165,0.193060,0.000147,<NA>,<NA>,<NA>
current_gnn,0.669516,0.005893,0.616166,0.007557,0.213097,0.003475,<NA>,<NA>,<NA>
gbdt_tuned,0.709561,0.004144,0.559467,0.002571,0.188549,0.001197,<NA>,<NA>,<NA>


In [4]:
per_seed

,seed,experiment,n_matches,accuracy,log_loss,brier,evaluation_hash
0,42,bscore_logit,1778.0,0.706974,0.577682,0.192962,5f30ac06efed318eead9834294bb42a6da67c1496a53af...
1,42,current_gnn,1778.0,0.666479,0.611591,0.210970,5f30ac06efed318eead9834294bb42a6da67c1496a53af...
2,42,antisymmetric_decoder,1778.0,0.706412,0.585474,0.199254,5f30ac06efed318eead9834294bb42a6da67c1496a53af...
3,42,gbdt_tuned,1778.0,0.710911,0.557241,0.187540,5f30ac06efed318eead9834294bb42a6da67c1496a53af...
4,123,bscore_logit,1778.0,0.708099,0.577712,0.192940,8ba7b0559c12e8e87fa1aedc59cac2ae9989001c53d89d...
5,123,current_gnn,1778.0,0.660855,0.627537,0.218278,8ba7b0559c12e8e87fa1aedc59cac2ae9989001c53d89d...
6,123,antisymmetric_decoder,1778.0,0.700225,0.586344,0.199735,8ba7b0559c12e8e87fa1aedc59cac2ae9989001c53d89d...
7,123,gbdt_tuned,1778.0,0.705287,0.557461,0.187646,8ba7b0559c12e8e87fa1aedc59cac2ae9989001c53d89d...
8,456,bscore_logit,1778.0,0.703600,0.578071,0.193303,027cea4306cd912a9994951c2e93f040adb7c735d59e55...
9,456,current_gnn,1778.0,0.674353,0.614749,0.212589,027cea4306cd912a9994951c2e93f040adb7c735d59e55...


In [5]:
per_seed.pivot(
    index="seed",
    columns="experiment",
    values=["accuracy", "log_loss", "brier"],
)

accuracy                                      \
experiment antisymmetric_decoder bscore_logit current_gnn gbdt_tuned   
seed                                                                   
42                      0.706412     0.706974    0.666479   0.710911   
123                     0.700225     0.708099    0.660855   0.705287   
456                     0.704162     0.703600    0.674353   0.705849   
789                     0.701912     0.709786    0.674916   0.710349   
2026                    0.699100     0.708099    0.670979   0.715411   

                        log_loss                                      \
experiment antisymmetric_decoder bscore_logit current_gnn gbdt_tuned   
seed                                                                   
42                      0.585474     0.577682    0.611591   0.557241   
123                     0.586344     0.577712    0.627537   0.557461   
456                     0.580378     0.578071    0.614749   0.558558   
789                     0.583875     0.577745    0.607920   0.563347   
2026                    0.581063     0.577918    0.619035   0.560729   

                           brier                                      
experiment antisymmetric_decoder bscore_logit current_gnn gbdt_tuned  
seed                                                                  
42                      0.199254     0.192962    0.210970   0.187540  
123                     0.199735     0.192940    0.218278   0.187646  
456                     0.197420     0.193303    0.212589   0.188421  
789                     0.198797     0.193010    0.209231   0.190519  
2026                    0.197634     0.193086    0.214415   0.188619

In [6]:
pivot = per_seed.pivot(
    index="seed",
    columns="experiment",
    values=["accuracy", "log_loss", "brier"],
)
for baseline in ["bscore_logit", "current_gnn", "antisymmetric_decoder"]:
    print("\nGBDT minus", baseline)
    display(pivot.xs("gbdt_tuned", level="experiment", axis=1)
            - pivot.xs(baseline, level="experiment", axis=1))


GBDT minus bscore_logit


,accuracy,log_loss,brier
seed,,,
42,0.003937,-0.020441,-0.005421
123,-0.002812,-0.020250,-0.005294
456,0.002250,-0.019513,-0.004882
789,0.000562,-0.014398,-0.002491
2026,0.007312,-0.017189,-0.004467



GBDT minus current_gnn


,accuracy,log_loss,brier
seed,,,
42,0.044432,-0.054350,-0.023429
123,0.044432,-0.070075,-0.030632
456,0.031496,-0.056191,-0.024168
789,0.035433,-0.044573,-0.018713
2026,0.044432,-0.058306,-0.025796



GBDT minus antisymmetric_decoder


,accuracy,log_loss,brier
seed,,,
42,0.004499,-0.028233,-0.011714
123,0.005062,-0.028883,-0.012089
456,0.001687,-0.021820,-0.008998
789,0.008436,-0.020529,-0.008278
2026,0.016310,-0.020334,-0.009015
